# CrossAttention

## CrossAttentionLayer

In [3]:
import torch
from torch import nn
from torch.nn import functional as F
import numpy as np


def cross_attention(q, k, v):
    d_k = q.size(-1)
    attn_scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(d_k)
    attn_weights = F.softmax(attn_scores, dim=-1)
    outputs = torch.matmul(attn_weights, v)
    return outputs, attn_weights

class CrossAttentionLayer(nn.Module):
    def __init__(self, d_model, d_k, d_v):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_k)
        self.W_k = nn.Linear(d_model, d_k)
        self.W_v = nn.Linear(d_model, d_v)

    def forward(self, enc_output, dec_output):
        Q = self.W_q(dec_output)
        K = self.W_k(enc_output)
        V = self.W_v(enc_output)
    
        output, attn_weights = cross_attention(Q, K, V)
        return output, attn_weights


# Проверка (без батчей)
Q = torch.tensor([[10, 0, 0, 0],
     [0, 10, 0, 0]]).float()
K = torch.tensor([[10, 0, 0, 0],
     [0, 10, 0, 0],
     [0, 0, 10, 0]]).float()
V = torch.tensor([[10, 0, 0, 0],
     [0, 20, 0, 0],
     [0, 0, 30, 0]]).float()
assert cross_attention(Q, K, V)[0].sum() == 30 

## EncoderDecoderWithAttention

In [4]:
class EncoderDecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, start_token_id, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.encoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.cross_attn = CrossAttentionLayer(d_model=hidden_dim, d_k=hidden_dim, d_v=hidden_dim)
        self.lm_head = nn.Linear(hidden_dim * 2, vocab_size)  # concat(dec_output, context)
        self.start_token_id = start_token_id
        self.max_len = max_len

    def forward(self, src, tgt=None):
        batch_size = src.size(0)

        # 1. Шаг энкодера
        embedded_src = self.embedding(src)                           # [B, T_src, E]
        encoder_outputs, (h, c) = self.encoder(embedded_src)          # enc_outputs: [B, T_src, H]

        # 2. Цикл декодера (без teacher forcing)
        input_token = torch.full((batch_size,), self.start_token_id,
                                 dtype=torch.long, device=src.device)

        logits_history = []
        attn_history = []
        dec_hidden, dec_cell = h, c

        for _ in range(self.max_len):
            embedded_t = self.embedding(input_token).unsqueeze(1)     # [B, 1, E]
            dec_output, (dec_hidden, dec_cell) = self.decoder(embedded_t, (dec_hidden, dec_cell))  # [B, 1, H]

            # CrossAttention
            context, attn = self.cross_attn(encoder_outputs, dec_output)   # [B, 1, H], [B, 1, T_src]
            attn_history.append(attn)

            # Конкатенируем decoder state + context
            concat_vec = torch.cat([dec_output, context], dim=-1)     # [B, 1, 2H]
            step_logits = self.lm_head(concat_vec)                    # [B, 1, vocab]
            logits_history.append(step_logits)

            # Берём токен с максимальной вероятностью
            input_token = step_logits.argmax(dim=-1).squeeze(1)       # [B]

        logits = torch.cat(logits_history, dim=1)                     # [B, max_len, vocab]
        attn_history = torch.cat(attn_history, dim=1)                 # [B, max_len, T_src]
        return logits, attn_history

def test_shapes():
    vocab_size = 50
    embed_dim = 16
    hidden_dim = 32
    start_token_id = 1
    max_len = 5

    model = EncoderDecoderWithAttention(vocab_size, embed_dim, hidden_dim,
                                        start_token_id=start_token_id,
                                        max_len=max_len)

    src = torch.randint(0, vocab_size, (2, 7))   # batch=2, src_len=7
    logits, attn = model(src)

    assert logits.shape == (2, max_len, vocab_size), f"Неправильный размер логитов: {logits.shape}"
    assert attn.shape == (2, max_len, src.size(1)), f"Неправильный размер весов внимания: {attn.shape}"
    print("Shapes test passed")


def test_greedy_generation():
    vocab_size = 10
    embed_dim = 8
    hidden_dim = 16
    start_token_id = 0
    max_len = 3

    model = EncoderDecoderWithAttention(vocab_size, embed_dim, hidden_dim,
                                        start_token_id=start_token_id,
                                        max_len=max_len)

    src = torch.randint(0, vocab_size, (1, 4))   # batch=1
    logits, attn = model(src)

    preds = logits.argmax(dim=-1)  # \[1, max_len\]
    print("Предсказанная последовательность:", preds.tolist())
    print("Веса внимания:\\n", attn)


# Run tests
test_shapes()
test_greedy_generation()

Shapes test passed
Предсказанная последовательность: [[5, 5, 5]]
Веса внимания:\n tensor([[[0.2552, 0.2497, 0.2477, 0.2474],
         [0.2560, 0.2486, 0.2477, 0.2477],
         [0.2563, 0.2483, 0.2476, 0.2477]]], grad_fn=<CatBackward0>)


## Teacher Forcing

In [5]:
class EncoderDecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, start_token_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.encoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.cross_attn = CrossAttentionLayer(d_model=hidden_dim, d_k=hidden_dim, d_v=hidden_dim)
        self.lm_head = nn.Linear(hidden_dim * 2, vocab_size)
        self.start_token_id = start_token_id

    def forward(self, src, tgt):
        batch_size, tgt_len = tgt.shape

        # 1. Энкодер
        embedded_src = self.embedding(src)
        encoder_outputs, (h, c) = self.encoder(embedded_src)

        # 2. Вход декодера
        start_tokens = torch.full((batch_size, 1), self.start_token_id,
                                  dtype=torch.long, device=src.device)  # [B, 1]
        decoder_inputs = torch.cat([start_tokens, tgt[:, :-1]], dim=1)  # shift right
        embedded_trg = self.embedding(decoder_inputs)                   

        # 3. Шаг декодера
        dec_output, (dec_hidden, dec_cell) = self.decoder(embedded_trg, (h, c))

        context, attn = self.cross_attn(encoder_outputs, dec_output)

        # 4. Линейный слой (LM Head)
        concat_vec = torch.cat([dec_output, context], dim=-1)
        logits = self.lm_head(concat_vec)
        return logits, attn

def test_teacher_forcing():
    vocab_size = 20
    embed_dim = 8
    hidden_dim = 16
    start_token_id = 0

    model = EncoderDecoderWithAttention(vocab_size, embed_dim, hidden_dim, start_token_id)

    src = torch.randint(0, vocab_size, (2, 5))   # batch=2, src_len=5
    tgt = torch.randint(0, vocab_size, (2, 6))   # batch=2, tgt_len=6

    logits, attn = model(src, tgt)

    # Check shapes
    assert logits.shape == (2, 6, vocab_size)
    assert attn.shape == (2, 6, src.size(1))
    print("Размерности логитов и весов внимания совпадают")


    # Check loss computation works
    criterion = nn.CrossEntropyLoss()
    loss = criterion(logits.view(-1, vocab_size), tgt.reshape(-1))
    print("Значение лосса:", loss.item())


# Run test
test_teacher_forcing()

Размерности логитов и весов внимания совпадают
Значение лосса: 3.01636004447937


# Предобучение T5

In [ ]:
import math
import random
from typing import List, Tuple

random.seed(42)

MASK_RATE = 0.15
MEAN_SPAN = 3.0

def sample_geometric(mean_span: float) -> int:
    """
    Сэмплирование длины спана из геометрического распределения.
    """
    p = 1.0 / mean_span
    span_len = int(math.ceil(math.log(1 - random.random()) / math.log(1 - p)))
    return max(1, span_len)

def span_corruption(tokens: List[str], mask_rate: float = MASK_RATE, mean_span: float = MEAN_SPAN) -> List[Tuple[int, int]]:
    """
    TODO: Верните список спанов для маскирования.
    Каждый спан — это (start, length).
    Маскируем ~mask_rate от общего числа токенов.
    """
    spans = []
    n = len(tokens)
    total_to_mask = max(1, int(round(n * mask_rate)))

    # ===== TODO =====
    covered = 0
    i = 0
    # Двигаемся слева направо и сэмплируем спаны, пока не наберется нужная доля
    while covered < total_to_mask and i < n:
        span_len = sample_geometric(mean_span)
        if i + span_len > n:
            span_len = n - i
        spans.append((i, span_len))
        covered += span_len
        i += span_len + 1  # оставляем зазор
    return spans

def prepare_pair(tokens: List[str], spans: List[Tuple[int, int]]) -> Tuple[str, str]:
    """
    TODO: Построить corrupted_input и target_output.
    Правила:
      - Во входе каждый спан заменяем на <extra_id_k>.
      - В выходе: <extra_id_k> + содержимое спана (все токены).
      - Сентинелы нумеруются слева направо.
    """
    corrupted = []
    target = []
    last_idx = 0
    sentinel_id = 0

    for start, length in spans:
        # Копируем токены до спана
        corrupted.extend(tokens[last_idx:start])
        sentinel = f"<extra_id_{sentinel_id}>"
        corrupted.append(sentinel)

        # Заполняем target: сентинел + вырезанный спан
        target.append(sentinel)
        target.extend(tokens[start:start+length])

        last_idx = start + length
        sentinel_id += 1

    # Добавляем хвост после последнего спана
    corrupted.extend(tokens[last_idx:])
    # Добавляем eos
    target.append("<eos>")

    corrupted_text = " ".join(corrupted)
    target_text = " ".join(target)
    return corrupted_text, target_text

# ===== Пример использования =====
tokens = "Модель T5 обучается с помощью span corruption".split()
spans = span_corruption(tokens, mask_rate=0.3)
inp, out = prepare_pair(tokens, spans)

print("Tokens :", tokens)
print("Spans  :", spans)
print("Input  :", inp)
print("Target :", out)

# Метрики seq2seq

## BLUE

In [7]:
import math
from collections import Counter

import math
from collections import Counter

def compute_bleu(candidate, reference, max_order=2):
    cand_tokens = candidate.split()
    ref_tokens = reference.split()

    precisions = []
    for n in range(1, max_order + 1):
        cand_ngrams = Counter([tuple(cand_tokens[i:i+n]) for i in range(len(cand_tokens) - n + 1)])
        ref_ngrams = Counter([tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1)])

        overlap = {ng: min(count, ref_ngrams[ng]) for ng, count in cand_ngrams.items()}
        p_n = sum(overlap.values()) / max(1, sum(cand_ngrams.values()))
        precisions.append(p_n)

    # Если хотя бы одна precision == 0, BLEU = 0
    if any(p == 0 for p in precisions):
        return 0.0

    # Brevity Penalty
    c, r = len(cand_tokens), len(ref_tokens)
    BP = 1 if c >= r else math.exp(1 - r / c)

    log_avg = sum(math.log(p) for p in precisions) / max_order
    bleu = BP * math.exp(log_avg)
    return bleu

candidate = "Ходор держал дверь"
reference = "Ходор закрыл дверь"


print("BLEU (ваша реализация): ", compute_bleu(candidate, reference))

import evaluate
reference_bleu = evaluate.load("bleu")
results = reference_bleu.compute(predictions=[candidate], references=[reference], tokenizer=lambda x: x.split(), max_order=2)
print("BLEU (референс): ", results["bleu"])

BLEU (ваша реализация):  0.0


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu130).


BLEU (референс):  0.0


## ROUGE

In [10]:
def lcs(X, Y):
    m, n = len(X), len(Y)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m):
        for j in range(n):
            if X[i] == Y[j]:
                dp[i+1][j+1] = dp[i][j] + 1
            else:
                dp[i+1][j+1] = max(dp[i][j+1], dp[i+1][j])
    return dp[m][n]

def rouge_l(candidate, reference):
    cand_tokens, ref_tokens = candidate.split(), reference.split()
    lcs_len = lcs(cand_tokens, ref_tokens)

    precision = lcs_len / len(cand_tokens)
    recall = lcs_len / len(ref_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

candidate, reference = "Ходор держал дверь", "Ходор держал дверь, чтобы Бран мог спастись"
print("Ваш Rouge-L: ", rouge_l(candidate, reference)[-1])

reference_rouge = evaluate.load('rouge')

print("Референсный Rouge-L: ", reference_rouge.compute(predictions=[candidate], references=[reference], tokenizer=lambda x: x.split())['rougeL'])

Ваш Rouge-L:  0.4


Референсный Rouge-L:  0.4


## BERTScore

In [11]:
import torch
from transformers import AutoTokenizer, AutoModel

# Загружаем модель
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
model = AutoModel.from_pretrained("cointegrated/rubert-tiny2")
model.eval()

def get_embeddings(text: str):
    """Возвращает эмбеддинги токенов без CLS/SEP."""
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    with torch.no_grad():
        outputs = model(**inputs)
    # Последний слой: [batch, seq_len, hidden_size]
    embeddings = outputs.last_hidden_state.squeeze(0)
    return embeddings

def bertscore_pair(hyp, ref):
    # Получаем эмбеддинги
    h = get_embeddings(hyp)   # [len_h, d]
    r = get_embeddings(ref)   # [len_r, d]

    # Нормировка эмбеддингов
    h = torch.nn.functional.normalize(h, p=2, dim=1)
    r = torch.nn.functional.normalize(r, p=2, dim=1)

    # Косинусное расстояние
    sim = torch.matmul(h, r.T)  # [len_h, len_r]

    # Precision: для каждого токена h берем max по r
    P = sim.max(dim=1).values.mean().item()
    # Recall: для каждого токена r берем max по h
    R = sim.max(dim=0).values.mean().item()
    # F1
    F1 = 2 * P * R / (P + R + 1e-8)

    return P, R, F1

# Пример
hyp = "Сегодня будет краткий дождь и прохладный ветер."
ref = "Сегодня ожидается непродолжительный дождь и прохладный ветер."

P, R, F1 = bertscore_pair(hyp, ref)
print(f"P={P:.4f}, R={R:.4f}, F1={F1:.4f}")

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


P=0.9045, R=0.9048, F1=0.9047


# Семантическое выравнивание предложений

In [14]:
import numpy as np
from scipy.spatial.distance import cosine
from sentence_transformers import SentenceTransformer
model_name = 'distiluse-base-multilingual-cased'
model_st = SentenceTransformer(model_name)

src = """Ветер свистел над стенами Винтерфелла, заставляя знамена домов дрожать в преддверии заката. Джон Сноу стоял на бастионе, наблюдая за тёмными лесами, где тени деревьев казались живыми. Он чувствовал надвигающуюся опасность, словно невидимые шаги Белых ходоков приближались к стенам крепости."""
trg = """The wind whistled over the walls of Winterfell, making the banners of the houses flutter in the approaching dusk. Jon Snow stood on the battlement, watching the dark forests where the shadows of the trees seemed alive. He felt the impending danger, as if the invisible footsteps of the White Walkers were drawing closer to the castle walls."""

srcs = src.split(".")
trgs = trg.split(".")

src_embeds = model_st.encode(srcs)
trg_embeds = model_st.encode(trgs)


def get_sim_matrix(a, b):
    sim_matrix = np.zeros((len(a), len(b)))
    for i in range(len(a)):
        for j in range(len(b)):
            sim = 1 - cosine(a[i], b[j])
            sim_matrix[i,j] = sim
    return sim_matrix

sim_matrix = get_sim_matrix(src_embeds, trg_embeds)

# проверяем результат
np.testing.assert_array_equal(sim_matrix.argmax(1), np.arange(len(src_embeds)))

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/607 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

# RAG

In [2]:
from typing import List, Dict, Union

import numpy as np
import torch

from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline, set_seed

from sklearn.metrics.pairwise import cosine_similarity


class RAG:

    def __init__(
            self,
            llm_name: str = "t-tech/T-lite-it-1.0",
            embedder_name: str = "cointegrated/rubert-tiny2",
            device: str = "cuda" if torch.cuda.is_available() else "cpu"):
        self.device = device

        # Инициализация модели для эмбеддингов
        self.embedder_tokenizer = AutoTokenizer.from_pretrained(embedder_name)
        self.embedder_model = AutoModel.from_pretrained(embedder_name).to(
            self.device)

        # Инициализация LLM
        self.llm_tokenizer = AutoTokenizer.from_pretrained(llm_name)
        self.llm_model = AutoModelForCausalLM.from_pretrained(llm_name).to(
            self.device)

        # База знаний: словарь с текстами и их эмбеддингами
        self.knowledge_base = {
            "texts": [],
            "embeddings": None  # Тензор с эмбеддингами
        }

    def get_embedding(self, text: str) -> torch.Tensor:
        """
        Получение эмбеддинга текста

        :param text: входной текст
        :return: эмбеддинг текста (тензор)
        """
        inputs = self.embedder_tokenizer(text,
                                         return_tensors="pt",
                                         padding="max_length",
                                         truncation=True,
                                         max_length=512).to(self.device)

        with torch.no_grad():
            outputs = self.embedder_model(**inputs)
            embeddings = outputs.last_hidden_state[:, 0, :]
            embeddings = torch.nn.functional.normalize(embeddings)

        return embeddings.cpu().numpy()

    def add_to_knowledge_base(self, text: str|List[str]) -> None:
        """
        Добавление текста в базу знаний

        :param text: текст для добавления
        """
        embedding = self.get_embedding(text)

        # Добавляем текст в список
        if isinstance(text, str):
            self.knowledge_base["texts"].append(text)
        else:
            self.knowledge_base["texts"].extend(text)

        # Обновляем тензор эмбеддингов
        if self.knowledge_base["embeddings"] is None:
            self.knowledge_base["embeddings"] = embedding
        else:
            self.knowledge_base["embeddings"] = np.vstack(
                [self.knowledge_base["embeddings"], embedding], dim=0)

    def find_closest(self,
                     query_embedding: np.ndarray,
                     top_k: int = 3) -> List[Dict]:
        """
        Поиск ближайших текстов в базе знаний

        :param query_embedding: эмбеддинг запроса
        :param top_k: количество возвращаемых результатов
        :return: список словарей с текстами и оценкой схожести
        """
        if len(self.knowledge_base["texts"]) == 0:
            return []

        # Вычисляем косинусную схожесть
        similarities = cosine_similarity(
            query_embedding.reshape(1, -1),
            self.knowledge_base["embeddings"])[0]

        # Получаем индексы топ-K результатов
        top_indices = similarities.argsort().argsort()[::-1][:min(top_k, len(similarities))]

        # Формируем результат
        results = []
        for idx in top_indices:
            results.append({
                "text": self.knowledge_base["texts"][idx],
                "score": similarities[idx]
            })

        return results

    def _create_prompt(self, question: str, context_texts: List[str]) -> str:
        """
        Создание промпта для LLM

        :param question: вопрос пользователя
        :param context_texts: список релевантных текстов из базы знаний
        :return: сформированный промпт
        """
        context = "\n\n".join([
            f"Контекст {i+1}: {text}" for i, text in enumerate(context_texts)
        ])

        prompt = f"""Используя приведённые ниже контексты, максимально кратко ответь на вопрос. Если в контекстах нет нужной информации, скажи об этом.

        {context}

        Вопрос: {question}"""

        return prompt

    def ask_question(self, question: str, top_k: int = 3) -> str:
        """
        Задание вопроса к системе RAG

        :param question: текст вопроса
        :param top_k: количество используемых контекстов из базы знаний
        :return: ответ модели
        """
        # Получаем эмбеддинг вопроса
        question_embedding = self.get_embedding(question)

        # Ищем релевантные тексты
        closest = self.find_closest(question_embedding, top_k=top_k)
        if len(closest) > 0:
            context_texts = [item["text"] for item in closest]
        else:
            context_texts = ["Релевантной информации не найдено"]

        # Создаём промпт
        prompt = self._create_prompt(question, context_texts)

        # Формируем сообщения для LLM
        messages = [{
            "role":
            "system",
            "content":
            "Ты виртуальный ассистент. Твоя задача - быть полезным диалоговым ассистентом."
        }, {
            "role": "user",
            "content": prompt
        }]

        # Генерируем ответ
        text = self.llm_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)

        model_inputs = self.llm_tokenizer([text],
                                          return_tensors="pt").to(self.device)

        generated_ids = self.llm_model.generate(**model_inputs,
                                                max_new_tokens=1024,
                                                do_sample=True,
                                                temperature=0.1)

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(
                model_inputs.input_ids, generated_ids)
        ]

        response = self.llm_tokenizer.batch_decode(generated_ids,
                                                   skip_special_tokens=True)[0]

        return response

In [3]:
rag = RAG()
db = ["X0Ja_asd - пароль от моего компьютера", "RisingTide - новая группа, состоящая из бывших моряков"]

# добавляем в БД информацию
rag.add_to_knowledge_base(db)

set_seed(42) # для воспроизводимости
answer = rag.ask_question("Я забыл пароль от своего компьютера")
print(answer)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/712 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

## Векторные базы данных

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch


model_name = "cross-encoder/stsb-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to("cuda")


query = "How to choose a laptop for work?"
candidates = [
"2023 rating of best laptops for office work",
"Comparison of Intel Core i5 vs i7 processors",
"How to improve performance of an old laptop",
"Optimal laptop specifications for programmers",
"Difference between SSD and HDD drives",
"10 common mistakes when buying a laptop",
"How to connect a laptop to a TV",
"Best budget laptops under 50,000 rubles",
"What graphics card is needed for graphic design work",
"How to extend laptop battery life",
]


def rerank(query, candidates):
    pairs = [(query, cand) for cand in candidates]
    inputs = tokenizer(pairs, return_tensors="pt", padding=True, truncation=True).to("cuda")
    with torch.no_grad():
        outputs = model(**inputs)
    scores = torch.sigmoid(outputs.logits).flatten().tolist()
    return sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)


reranked = rerank(query, candidates)
print(reranked)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu130).


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]